# Code Lab: Ensemble Learning & Model Evaluation

In this lab, we will load a real-world dataset (Breast Cancer Wisconsin), train a single Decision Tree as a baseline, and then prove mathematically why **Random Forests (Bagging)** and **XGBoost (Boosting)** are vastly superior.

We will also plot an **ROC Curve** to visually compare their performances.

In [ ]:
# 1. Install necessary libraries (XGBoost)
!pip install xgboost scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc

print("Libraries imported successfully!")

### Step 1: Data Preparation
We load the dataset and split it into training (80%) and testing (20%) sets.

In [ ]:
# Load the dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target # 0: Malignant, 1: Benign

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training instances: {len(X_train)} | Testing instances: {len(X_test)}")

### Step 2: The Baseline (Single Decision Tree)
A single decision tree is highly interpretable but prone to overfitting and high variance.

In [ ]:
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

tree_preds = tree_model.predict(X_test)
tree_probs = tree_model.predict_proba(X_test)[:, 1] # Probability of Class 1

print(f"Decision Tree Accuracy: {accuracy_score(y_test, tree_preds):.4f}")

### Step 3: Bagging (Random Forest)
We now train 100 decision trees in parallel using Bootstrapping. They will vote on the final outcome.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_preds):.4f}")

### Step 4: Boosting (XGBoost)
We train sequentially, where each tree aims to fix the errors made by the previous trees.

In [ ]:
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

print(f"XGBoost Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")

### Step 5: Advanced Model Evaluation (ROC Curve)
Accuracy is often misleading. The **Receiver Operating Characteristic (ROC)** curve plots the True Positive Rate against the False Positive Rate. The higher the Area Under the Curve (AUC), the better!

In [ ]:
plt.figure(figsize=(10, 8))

# Calculate ROC coordinates for each model
models_probs = {
    'Decision Tree': tree_probs,
    'Random Forest': rf_probs,
    'XGBoost': xgb_probs
}

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for (name, probs), color in zip(models_probs.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

# Plot the baseline (random guessing)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label="Random Guessing (AUC = 0.500)")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison: Single Tree vs Ensembles')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

### Conclusion
By looking at the ROC plot, it is visually obvious that Ensembles (Random Forest & XGBoost) massively outperform a single Decision Tree. Their curves push much closer to the top-left corner, meaning they catch more true positives while making fewer false positive mistakes.